# Prefect workflow for running the s3l0 eopf processor with the rs-dpr-service

https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-652

See the associated:

  * Python module: [s3l0_demo_processor.py](./s3l0_demo_processor.py)
  * YAML file: [s3l0_demo_processor.yaml](./s3l0_demo_processor.yaml)

## 1. Initialisation

In [1]:
import os
print(f"Prefect server URL used internally: {os.environ['PREFECT_API_URL']}")
dashboard = f"{os.environ['RSPY_PREFECT_URL']}/dashboard"
print(f"Prefect dashboard public URL: {dashboard}")

Prefect server URL used internally: http://prefect-server:4200/api
Prefect dashboard public URL: http://localhost:4200/dashboard


In [2]:
# Init environment before running a demo notebook.
from resources.utils import *  
from resources.dask_utils import *
USE_DPR_MOCKUP = True
if os.getenv("RSPY_LOCAL_MODE") == "1" and USE_DPR_MOCKUP:
    os.environ["DASK_GATEWAY_EOPF_ADDRESS"] = os.environ["DASK_GATEWAY_EOPF_MOCKUP_ADDRESS"]
    os.environ["DASK_GATEWAY_EOPF_PUBLIC"] = os.environ["DASK_GATEWAY_EOPF_MOCKUP_PUBLIC"]

init_demo()
init_dask_cluster_eopf(scale=2, use_mockup = USE_DPR_MOCKUP)
init_dask_cluster_staging(scale=2)

# Reload the global vars again
from resources.utils import *  
from resources.dask_utils import *  

# You can check here the number of workers, threads and memory per worker.
# In local mode, you can configure them by running e.g.
# DASK_MEMORY_EOPF=4G DASK_THREADS_EOPF=4 docker compose up # ...
display(dask_cluster_eopf)
display(dask_cluster_staging)

DependencyConflict: requested: "starlette >= 0.13, <0.15" but found: "starlette 0.46.2"


Auxip service: http://rs-server-adgs:8000/auxip
CADIP service: http://rs-server-cadip:8000/cadip
Catalog service: http://rs-server-catalog:8000
Staging service: http://rs-server-staging:8000
DPR service: http://rs-dpr-service:8000
Connecting to dask gateway for 'dask-eopf-mockup': http://dask-eopf-mockup:8000 ...
Create new dask cluster
Dask dashboard for 'dask-eopf-mockup': http://localhost:8703/clusters/069aa74395bf41c09c83e6347d1227b8/status


/opt/conda/lib/python3.11/site-packages/distributed/client.py:1394: VersionMismatchWarning: Mismatched versions found

+-------------+----------+-----------+---------+
| Package     | Client   | Scheduler | Workers |
+-------------+----------+-----------+---------+
| dask        | 2024.5.2 | 2025.2.0  | None    |
| distributed | 2024.5.2 | 2025.2.0  | None    |
| tornado     | 6.3.3    | 6.4.2     | None    |
+-------------+----------+-----------+---------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))


Dask workers for 'dask-eopf-mockup' are up: 0/2
Dask workers for 'dask-eopf-mockup' are up: 2/2
Connecting to dask gateway for 'dask-staging': http://dask-staging:8000 ...
Create new dask cluster
Dask dashboard for 'dask-staging': http://localhost:8701/clusters/972cb12be1464ca09b607a010c2ac0c4/status


/opt/conda/lib/python3.11/site-packages/distributed/client.py:1394: VersionMismatchWarning: Mismatched versions found

+---------+--------+-----------+---------+
| Package | Client | Scheduler | Workers |
+---------+--------+-----------+---------+
| tornado | 6.3.3  | 6.4.2     | None    |
+---------+--------+-----------+---------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))


Dask workers for 'dask-staging' are up: 0/2
Dask workers for 'dask-staging' are up: 2/2


In [3]:
# Create a test collection
TEST_COLLECTION_NAME = "RSPY_643_TEST_COLLECTION"
collection = create_test_collection(TEST_COLLECTION_NAME)

# Check the catalog for RSPY_643_TEST_COLLECTION
items = catalog_client.get_items(TEST_COLLECTION_NAME)
assert not list(items)

#CADIP_SESSION_FILTER = "id=S3A_20250109134406046340" # Session id "platform='sentinel-1a'" "id=S1A_20200105072204051312" S3A_20250109134406046340 | S1A_20200105072204051312
CADIP_SESSION_FILTER ="id=S1A_20200105072204051312"


10:19:47.003 [INFO] (rs_client.rs_client) Retrieving all items from collection 'jgaucher:RSPY_643_TEST_COLLECTION'.


In [4]:
# Other imports
import os
import os.path as osp
from rs_common import prefect_utils
from rs_common.prefect_utils import *

# Get the prefect share bucket folder
share_bucket = await get_share_bucket()

# s3 bucket dirs that will contain the data
s3_base = osp.join(
    "s3://",
    share_bucket.bucket_name,
    share_bucket.bucket_folder,
    "users",
    OWNER_ID,
    "l0",
)
s3_config = osp.join(s3_base, "config")
s3_output = osp.join(s3_base, "output")

# Upload the local configuration dir to s3 bucket
await s3_upload_dir("./l0/config", s3_config)

flow_parameters = {
    "dask_eopf_name": dask_cluster_eopf.name, 
    "input_config_dir": s3_config,
    "payload_file": "s3/s3_l0_demo_payload_dpr_mockup_template.yaml",
    "output_data_dir": f"{s3_output}/s3",
    "owner_id": OWNER_ID,
    "collection_name": TEST_COLLECTION_NAME,
    "cadip_stac_filter": CADIP_SESSION_FILTER,
    "staging_timeout": 120,
    "use_dpr_mockup": USE_DPR_MOCKUP,
}

# Convert to json to trigger prefect flow
def to_json(my_data):
    return json.dumps(my_data).replace('"', r'\"')

10:19:47.232 | INFO    | prefect.S3Bucket - Uploading from 'l0/config/logging_config.yaml' to the bucket 'rs-dev-cluster-temp' path 'prefect-share/users/jgaucher/l0/config/logging_config.yaml'.

10:19:47.234 | INFO    | prefect.S3Bucket - Uploading from 'l0/config/s3/s3_l0_demo_payload_dpr_mockup_template.yaml' to the bucket 'rs-dev-cluster-temp' path 'prefect-share/users/jgaucher/l0/config/s3/s3_l0_demo_payload_dpr_mockup_template.yaml'.

10:19:47.235 | INFO    | prefect.S3Bucket - Uploading from 'l0/config/s3/l0_processor_configuration_3A.yaml' to the bucket 'rs-dev-cluster-temp' path 'prefect-share/users/jgaucher/l0/config/s3/l0_processor_configuration_3A.yaml'.

10:19:47.236 | INFO    | prefect.S3Bucket - Uploading from 'l0/config/s3/l0_processor_configuration_dpr_mockup.yaml' to the bucket 'rs-dev-cluster-temp' path 'prefect-share/users/jgaucher/l0/config/s3/l0_processor_configuration_dpr_mockup.yaml'.

10:19:47.257 | INFO    | prefect.S3Bucket - Uploaded 4 files from 'l0/config' to the bucket 'rs-dev-cluster-temp' path 'prefect-share/users/jgaucher/l0/config/s3/l0_processor_configuration_dpr_mockup.yaml'

In [5]:
# Save cluster info to be read by our flow
os.environ["DASK_CLUSTER_EOPF_NAME"] = dask_cluster_eopf.name
os.environ["DASK_CLUSTER_STAGING_NAME"] = dask_cluster_staging.name
if cluster_mode:
    os.environ["DASK_GATEWAY_EOPF_ADDRESS"] = os.environ["DASK_GATEWAY_ADDRESS"]

# Setup adaptive scaling
#dask_gateway.adapt_cluster(dask_cluster.name, minimum=1, maximum=scale)

## 2. Deploy Prefect flow

We deploy our source code via the S3 bucket.

In [6]:
# Use a subfolder named after the current user
s3_code_folder = f"users/{OWNER_ID}/code" 

if local_mode:
    print (f"S3 MinIO dashboard: http://localhost:9101 with user=minio password=Strong#Pass#1234")
print(f"Upload local source code to: 's3://{share_bucket.bucket_name}/{share_bucket.bucket_folder}/{s3_code_folder}'")

# Upload local directory and resources contents
await share_bucket.put_directory(local_path = ".", to_path = s3_code_folder)
await share_bucket.put_directory(local_path = "../../resources", to_path = f"{s3_code_folder}/resources")

# Pass the full S3 code folder as an environment variable
os.environ["S3_CODE_FOLDER"] = f"{share_bucket.bucket_folder}/{s3_code_folder}"

S3 MinIO dashboard: http://localhost:9101 with user=minio password=Strong#Pass#1234
Upload local source code to: 's3://rs-dev-cluster-temp/prefect-share/users/jgaucher/code'


In [7]:
%%bash
# Deploy the flow. We don't need to be in the git root folder.
prefect --no-prompt deploy --prefect-file "./s3l0_demo_processor_with_dpr_service.yaml"

10:19:49.502 | ERROR   | opentelemetry.instrumentation.instrumentor - DependencyConflict: requested: "starlette >= 0.13, <0.15" but found: "starlette 0.46.2"


╭──────────────────────────────────────────────────────────────────────────────╮
│ Deployment 's3l0-demo-processor/sprint23-s3l0-demo-processor' successfully   │
│ created with id '82c48de1-e942-4fb3-8251-22fafb3cec2e'.                      │
╰──────────────────────────────────────────────────────────────────────────────╯

View Deployment in UI: http://prefect-server:4200/deployments/deployment/82c48de1-e942-4fb3-8251-22fafb3cec2e


To schedule a run for this deployment, use the following command:

        $ prefect deployment run 
's3l0-demo-processor/sprint23-s3l0-demo-processor'



In [8]:
deploy_name = "s3l0-demo-processor/sprint23-s3l0-demo-processor"
await prefect_utils.wait_for_deployment(deploy_name)

Finished deploying prefect flow: 's3l0-demo-processor/sprint23-s3l0-demo-processor'


## 3. Run Prefect flow

In [9]:
output_data_dir = flow_parameters["output_data_dir"]
print(f"Remove existing zarr products from: {output_data_dir!r}")
s3_delete(output_data_dir)

# Convert to json to trigger prefect flow
params_str = to_json(flow_parameters) # flow parameters

Remove existing zarr products from: 's3://rs-dev-cluster-temp/prefect-share/users/jgaucher/l0/output/s3'


In [10]:
%%bash -s "$deploy_name" "$params_str"
# Trigger a run for this flow from the command line
prefect deployment run "$1" --params "$2" --watch

Creating flow run for deployment 
's3l0-demo-processor/sprint23-s3l0-demo-processor'...
Created flow run 'burgundy-mustang'.
└── UUID: ced87f78-0b0a-4ef8-9894-1609aaf35334
└── Parameters: {'dask_eopf_name': '069aa74395bf41c09c83e6347d1227b8', 'input_config_dir': 's3://rs-dev-cluster-temp/prefect-share/users/jgaucher/l0/config', 'payload_file': 's3/s3_l0_demo_payload_dpr_mockup_template.yaml', 'output_data_dir': 's3://rs-dev-cluster-temp/prefect-share/users/jgaucher/l0/output/s3', 'owner_id': 'jgaucher', 'collection_name': 'RSPY_643_TEST_COLLECTION', 'cadip_stac_filter': 'id=S1A_20200105072204051312', 'staging_timeout': 120, 'use_dpr_mockup': True}
└── Job Variables: {}
└── Scheduled start time: 2025-05-22 10:19:55 UTC (now)
└── URL: http://prefect-server:4200/runs/flow-run/ced87f78-0b0a-4ef8-9894-1609aaf35334
Watching flow run 'burgundy-mustang'...


10:19:59.976 | INFO    | prefect - Flow run is in state 'Pending'
10:20:03.464 | INFO    | prefect - Flow run is in state 'Running'
10:20:27.866 | INFO    | prefect - Flow run is in state 'Failed'


Flow run finished in state 'Failed'.


CalledProcessError: Command 'b'# Trigger a run for this flow from the command line\nprefect deployment run "$1" --params "$2" --watch\n'' returned non-zero exit status 1.

In [ ]:
print(f"Output products generated on: {output_data_dir!r}")

local_report_dir = osp.join("./l0", "reports", "s1.short")
print(f"Download reports locally: {local_report_dir!r}")
await s3_download_dir(osp.join(output_data_dir, "reports"), local_report_dir)
eopf_prod_ids = ["S03MWRL0__20221101T092439_6037_A307_T677", "S03OLCL0__20210629T044945_0119_A247_T219"]
for id in eopf_prod_ids:
    assert catalog_client.get_item(TEST_COLLECTION_NAME, id) 
   

## 6. Shutdown the dask clusters

In [ ]:
shutdown = False
if shutdown:    
    # You can scale the clusters to 0 workers
    dask_gateway_eopf.scale_cluster(dask_cluster_eopf.name, 0)
    dask_gateway_staging.scale_cluster(dask_cluster_staging.name, 0)

    # Or shutdown the clusters
    shutdown_dask_clusters(dask_gateway_eopf, dask_cluster_eopf.name)
    shutdown_dask_clusters(dask_gateway_staging, dask_cluster_staging.name)

    # Close the python objects
    close_dask_clusters()

# NOTE: restart your python kernel or terminal after the shutdown
# to avoid strange behaviour.

## For testing only: reset the cluster and run the flow locally from Python

In [ ]:
from importlib import reload
debug_flow = False

In [ ]:
if debug_flow:
    shutdown_dask_clusters(dask_gateway_staging, None)
    shutdown_dask_clusters(dask_gateway_eopf, None)
    init_dask_cluster_eopf(scale=2)
    init_dask_cluster_staging(scale=2)

    from resources.dask_utils import *
    flow_parameters["dask_eopf_name"] = dask_cluster_eopf.name

In [ ]:
if debug_flow:
    import s3l0_demo_processor
    reload(s3l0_demo_processor)
    results = s3l0_demo_processor.s3l0_demo_processor(**flow_parameters)
    display(results)